In [2]:
print(123)

123


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
len(documents)

72

In [6]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [7]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
doc = documents[:3]

In [14]:
doc

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [15]:
import json

user_prompt = json.dumps(doc)

In [16]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [18]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [19]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [22]:
from evaluation_utils import llm_structured

/workspaces/llm-zoomcamp-2026-code/w4_hw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What problem does RAG solve that a plain language model can’t handle well?', 'Why do we need to store the API key in a .env file instead of hardcoding it?', 'What tools and packages do I need to set up before starting this module?', 'How does adding FAQ text to the prompt make the model answer course questions better?', 'What are the main parts of a RAG system, and how do they work together?']


In [24]:
usage.input_tokens, usage.output_tokens

(3731, 101)

**Q1:** What's the average number of input tokens across these 3 calls?  
**Answer:** 1400

In [25]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [29]:
len(chunks)

295

In [26]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [34]:
from minsearch import VectorSearch, Index
import numpy as np
from embedder import Embedder

embed = Embedder()

2026-07-09 20:28:13.343007231 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [35]:
chunk_contents = [chunk['content'] for chunk in chunks]
chunk_embeddings = np.vstack([embed.encode(text) for text in chunk_contents])
X = chunk_embeddings

In [45]:
text_index = Index(text_fields=["content"])
text_index.fit(chunks)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [87]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    query_embedding = embed.encode(query)
    return vector_index.search(query_embedding, num_results=num_results)

In [88]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=5)
    vector_results = vector_search(query, num_results=5)
    return rrf([text_results, vector_results], k=k)

In [89]:
query3 = "How do I give the model access to tools?"
hybrid_results3 = hybrid_search(query3, k=60)
hybrid_results3

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

In [90]:
import pandas as pd
import os

csv_path = "data/ground-truth.csv"

df_ground_truth = pd.read_csv(csv_path)

    
ground_truth = df_ground_truth.to_dict(orient="records")


In [91]:
q = ground_truth[0]
q

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [92]:
text_results = text_search(q["question"], num_results=10)

text_results[:5]

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

**Q2:** After running text_search for it, what's the filename of the first result?  
**Answer:** 01-agentic-rag/lessons/03-rag.md

In [93]:
vector_results = vector_search(q["question"], num_results=10)
vector_results[:5]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

**Q3:** After running vector_search for the same question, what's the filename of the first result?  
**Answer:** 01-agentic-rag/lessons/01-intro.md

In [94]:
ground_truth[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [95]:
for d in text_results[:5]:
    print(f'{d["filename"]} == {ground_truth[0]["filename"]}: {d["filename"] == ground_truth[0]["filename"]}')

01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/01-intro.md == 01-agentic-rag/lessons/01-intro.md: True


In [96]:
def compute_relevance_text(q):
    doc_name = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_name))

    return relevance

In [97]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[0, 0, 0, 0, 1]

In [110]:
def compute_relevance(q, search_function, k=60):
    doc_name = q["filename"]
    results = search_function(query=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_name))

    return relevance

In [112]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function, k=60):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, k)
        relevance_total.append(relevance)

    return relevance_total

In [113]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [101]:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total

100%|██████████| 360/360 [00:00<00:00, 598.28it/s]


[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 1, 1, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 1, 0],
 [1, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0,

In [102]:
hit_rate(relevance_total)

0.7583333333333333

**Q4:** What's the Hit Rate?  
**Answer:** 0.7583333333333333

In [103]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [114]:
def compute_relevance(q, search_function, k=60):
    doc_name = q["filename"]

    if search_function is hybrid_search:
        results = search_function(query=q["question"], k=k)
    else:
        results = search_function(query=q["question"], num_results=k)

    return [int(d["filename"] == doc_name) for d in results]


  0%|          | 0/360 [00:00<?, ?it/s]


TypeError: vector_search() got an unexpected keyword argument 'k'

In [105]:
mrr(relevance_total)

0.5486111111111112

**Q5:** What's the MRR?  
**Answer:** 0.5486111111111112

In [106]:
relevance_total = compute_relevance_total(ground_truth, hybrid_search)


100%|██████████| 360/360 [00:05<00:00, 65.04it/s]


In [109]:
mrr(relevance_total)

0.6467592592592594